# Introducere în backtesting cu vectorbt

**Backtesting** = rulăm o regulă de tranzacționare pe date istorice ca să vedem cum *s-ar fi* comportat. Nu ne spune cum se va comporta în viitor, dar ne arată rapid dacă o idee are vreo șansă și, mai ales, ne ajută să prindem greșelile din ea.

În acest notebook:
1. descărcăm prețurile zilnice SPY (ETF-ul pe S&P 500);
2. construim **3 strategii simple**: trend (medii mobile, long/short), mean-reversion (RSI, doar long) și benzi Bollinger (long/short);
3. le comparăm cu **buy & hold**, folosind statisticile și graficele din vectorbt;
4. optimizăm parametrii și vedem cum arată **overfitting-ul**;
5. vedem două greșeli clasice care fac un backtest să pară mult mai bun decât este.

> **Google Colab:** *File → Upload notebook*, apoi *Runtime → Run all*. Prima celulă instalează pachetele lipsă (durează ~1 minut).

In [1]:
import sys
try:
    import vectorbt, yfinance
except ImportError:
    %pip install -q vectorbt yfinance

In [2]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import yfinance as yf
import vectorbt as vbt
import plotly.io as pio

if "google.colab" not in sys.modules:
    pio.renderers.default = "plotly_mimetype"   # local Jupyter / VS Code
vbt.settings.plotting["use_widgets"] = False    # plain figures render everywhere, incl. Colab
vbt.settings.returns["year_freq"] = "252 days"  # annualise with trading days, not calendar days
print("vectorbt", vbt.__version__)

vectorbt 1.2.0


## 1 · Datele

Folosim prețuri **ajustate** (`auto_adjust=True`): dividendele și split-urile sunt incluse în preț, deci randamentul este randamentul total pe care l-ar fi primit un investitor.

Împărțim istoricul în două:
- **in-sample** (2010–2019): aici avem voie să „ne jucăm” cu parametrii;
- **out-of-sample** (2020 → azi): date pe care strategia nu le-a „văzut” niciodată, ca un examen final.

In [3]:
TICKER = "SPY"
raw = yf.download(TICKER, start="2010-01-01", auto_adjust=True, progress=False)
if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)
ohlcv = raw[["Open", "High", "Low", "Close", "Volume"]].dropna()
ohlcv.index = ohlcv.index.tz_localize(None)

close, open_ = ohlcv["Close"], ohlcv["Open"]
IS_END = "2019-12-31"
in_sample, out_sample = close.index <= IS_END, close.index > IS_END
print(f"{TICKER}: {len(close)} zile, {close.index[0].date()} → {close.index[-1].date()}")
ohlcv.tail()

SPY: 4236 zile, 2010-01-04 → 2026-11-04


Price,Open,High,Low,Close,Volume
Date,,,,,
2026-10-29,768.789978,770.760010,766.619995,769.359985,40299600
2026-10-30,761.380005,761.460022,750.359985,755.719971,77696500
2026-11-02,759.450012,763.859985,751.599976,752.099976,50804300
2026-11-03,752.729980,757.070007,752.390015,754.349976,32891600
2026-11-04,762.989990,769.219971,762.640015,767.609985,40416577


In [4]:
ohlcv.loc["2025-06":].vbt.ohlcv.plot(title=f"{TICKER} — lumânări și volum (ultimele luni)").show()

## 2 · Cum funcționează un backtest

```
prețuri ──► indicator ──► semnale (True/False) ──► ordine ──► portofoliu ──► statistici
            (MA, RSI)     (intrare / ieșire)       (când, cât,  (cash, poziții,
                                                     la ce preț)  valoare zilnică)
```

Trei reguli pe care le respectăm peste tot:
1. **Fără privit în viitor.** Un semnal calculat cu prețul de *închidere* de azi poate fi executat abia **mâine la deschidere**. De aceea decalăm semnalele cu o zi (`shift(1)`) și folosim prețul `Open` pentru execuție.
2. **Costuri.** Fiecare tranzacție plătește comision și slippage (diferența dintre prețul vrut și cel obținut). Aici: 0,05% + 0,05%.
3. **Reper (benchmark).** O strategie e interesantă doar dacă bate ceva simplu, de exemplu buy & hold, după ajustarea la risc.

Funcția de mai jos face exact asta și o refolosim pentru toate strategiile.

In [5]:
FEES, SLIPPAGE, INIT_CASH = 0.0005, 0.0005, 10_000

def backtest(entries, exits, short_entries=None, short_exits=None, mask=None):
    # Signals are known at today's close -> trade at tomorrow's open.
    def prep(sig):
        if sig is None:
            return None
        sig = sig.shift(1, fill_value=False)
        return sig[mask] if mask is not None else sig
    px_close = close[mask] if mask is not None else close
    px_open = open_[mask] if mask is not None else open_
    return vbt.Portfolio.from_signals(
        px_close, entries=prep(entries), exits=prep(exits),
        short_entries=prep(short_entries), short_exits=prep(short_exits),
        price=px_open, fees=FEES, slippage=SLIPPAGE, init_cash=INIT_CASH, freq="1D")

benchmark = vbt.Portfolio.from_holding(close, fees=FEES, init_cash=INIT_CASH, freq="1D")
benchmark.stats()

Start                         2010-01-04 00:00:00
End                           2026-11-04 00:00:00
Period                         4236 days 00:00:00
Start Value                               10000.0
End Value                            90937.037193
Total Return [%]                       809.370372
Benchmark Return [%]                   809.825057
Max Gross Exposure [%]                      100.0
Total Fees Paid                          4.997501
Max Drawdown [%]                        33.717267
Max Drawdown Duration           488 days 00:00:00
Total Trades                                    1
Total Closed Trades                             0
Total Open Trades                               1
Open Trade PnL                       80937.037193
Win Rate [%]                                  NaN
Best Trade [%]                                NaN
Worst Trade [%]                               NaN
Avg Winning Trade [%]                         NaN
Avg Losing Trade [%]                          NaN


Câteva statistici de reținut:
- **Total Return**: câștigul total; **Benchmark Return**: cât a făcut activul în sine.
- **Max Drawdown**: cea mai mare scădere de la un vârf la un minim. E durerea maximă pe care ai fi suportat-o.
- **Sharpe Ratio**: randament mediu / volatilitate, anualizat. Peste 1 e bun; peste 2 pe acțiuni, la frecvență zilnică, e suspect.
- **Win Rate** / **Profit Factor**: procentul tranzacțiilor câștigătoare / câștiguri totale împărțite la pierderi totale.

## 3 · Strategia 1: încrucișarea mediilor mobile (trend, long/short)

**Ideea:** când media pe 50 de zile trece **peste** media pe 200 de zile, piața e pe trend ascendent → **long**. Când trece **sub** → **short** (pariem pe scădere). Strategia este mereu în piață.

In [6]:
fast = vbt.MA.run(close, 50, short_name="fast")
slow = vbt.MA.run(close, 200, short_name="slow")
golden = fast.ma_crossed_above(slow)    # golden cross -> go long
death = fast.ma_crossed_below(slow)     # death cross  -> go short

fig = close.vbt.plot(trace_kwargs=dict(name="Close"))
fast.ma.vbt.plot(trace_kwargs=dict(name="MA 50"), fig=fig)
slow.ma.vbt.plot(trace_kwargs=dict(name="MA 200"), fig=fig)
golden.vbt.signals.plot_as_entry_markers(close, fig=fig)
death.vbt.signals.plot_as_exit_markers(close, fig=fig)
fig.update_layout(title="MA 50/200: triunghi verde = long, roșu = short").show()

In [7]:
pf_ma = backtest(entries=golden, exits=death, short_entries=death, short_exits=golden)
pf_ma.plot(subplots=["orders", "trade_pnl", "cum_returns", "drawdowns"],
           title="Strategia 1 — MA crossover long/short").show()
pf_ma.stats()

Start                              2010-01-04 00:00:00
End                                2026-11-04 00:00:00
Period                              4236 days 00:00:00
Start Value                                    10000.0
End Value                                 17138.332278
Total Return [%]                             71.383323
Benchmark Return [%]                        809.825057
Max Gross Exposure [%]                           100.0
Total Fees Paid                             172.642769
Max Drawdown [%]                             48.763662
Max Drawdown Duration               1014 days 00:00:00
Total Trades                                        14
Total Closed Trades                                 13
Total Open Trades                                    1
Open Trade PnL                             3525.391961
Win Rate [%]                                 38.461538
Best Trade [%]                                57.93431
Worst Trade [%]                             -27.981044
Avg Winnin

Fiecare tranzacție în parte: verde = câștig, roșu = pierdere. Observați că în trendurile lungi câștigă mult, iar în piețele fără direcție dă multe semnale false (whipsaw).

In [8]:
pf_ma.trades.plot(title='Tranzacțiile strategiei MA').show()

## 4 · Strategia 2: RSI mean-reversion (doar long)

**Ideea:** după o scădere bruscă, prețul tinde să revină („mean reversion”). **RSI** (Relative Strength Index, 0–100) măsoară cât de puternică a fost mișcarea recentă. Cumpărăm când RSI(14) coboară **sub 30** (supravândut) și vindem când urcă **peste 55**. Între tranzacții stăm în cash.

In [9]:
rsi = vbt.RSI.run(close, window=14)
rsi_entries = rsi.rsi_crossed_below(30)
rsi_exits = rsi.rsi_crossed_above(55)
rsi.plot(levels=(30, 55)).update_layout(title="RSI(14) cu pragurile 30 / 55").show()

In [10]:
pf_rsi = backtest(entries=rsi_entries, exits=rsi_exits)
pf_rsi.plot(subplots=["orders", "trade_pnl", "cum_returns", "underwater"],
            title="Strategia 2 — RSI mean-reversion").show()
pf_rsi.stats()

Start                             2010-01-04 00:00:00
End                               2026-11-04 00:00:00
Period                             4236 days 00:00:00
Start Value                                   10000.0
End Value                                32518.772925
Total Return [%]                           225.187729
Benchmark Return [%]                       809.825057
Max Gross Exposure [%]                          100.0
Total Fees Paid                             959.07378
Max Drawdown [%]                             28.90327
Max Drawdown Duration               438 days 00:00:00
Total Trades                                       51
Total Closed Trades                                51
Total Open Trades                                   0
Open Trade PnL                                    0.0
Win Rate [%]                                80.392157
Best Trade [%]                              12.495926
Worst Trade [%]                             -12.39038
Avg Winning Trade [%]       

Strategia stă mult timp în cash, așa că randamentul total e mai mic, dar și expunerea la risc este mai mică. Comparați **Max Drawdown** și numărul de zile în piață cu buy & hold.

## 5 · Strategia 3: benzi Bollinger (mean-reversion, long/short)

**Benzile Bollinger** = media mobilă pe 20 de zile ± 2 deviații standard. Dacă prețul iese **sub** banda de jos, considerăm că a scăzut „prea mult” → **long**. Dacă iese **peste** banda de sus → **short**. Închidem poziția când prețul revine la mijloc (la medie).

In [11]:
bb = vbt.BBANDS.run(close, window=20, alpha=2)
bb_long_in = close.vbt.crossed_below(bb.lower)
bb_long_out = close.vbt.crossed_above(bb.middle)
bb_short_in = close.vbt.crossed_above(bb.upper)
bb_short_out = close.vbt.crossed_below(bb.middle)

bb.loc["2024":].plot().update_layout(title="Benzi Bollinger (20, 2σ) — 2024 → azi").show()
pf_bb = backtest(entries=bb_long_in, exits=bb_long_out,
                 short_entries=bb_short_in, short_exits=bb_short_out)
pf_bb.plot(subplots=["orders", "trade_pnl", "cum_returns", "drawdowns"],
           title="Strategia 3 — Bollinger long/short").show()
pf_bb.stats()

Start                             2010-01-04 00:00:00
End                               2026-11-04 00:00:00
Period                             4236 days 00:00:00
Start Value                                   10000.0
End Value                                14420.364404
Total Return [%]                            44.203644
Benchmark Return [%]                       809.825057
Max Gross Exposure [%]                          100.0
Total Fees Paid                           1915.976377
Max Drawdown [%]                            31.941125
Max Drawdown Duration              1862 days 00:00:00
Total Trades                                      154
Total Closed Trades                               153
Total Open Trades                                   1
Open Trade PnL                             139.023077
Win Rate [%]                                66.013072
Best Trade [%]                               6.620479
Worst Trade [%]                             -12.39038
Avg Winning Trade [%]       

## 6 · Comparația celor trei strategii

Aceeași perioadă, aceleași costuri, același capital inițial.

In [12]:
pfs = {"Buy & hold": benchmark, "MA 50/200 L/S": pf_ma, "RSI long": pf_rsi, "Bollinger L/S": pf_bb}
pd.DataFrame({k: pf.value() for k, pf in pfs.items()}).vbt.plot(
    title="Evoluția a 10.000 $").show()
pd.DataFrame({k: pf.drawdown() * 100 for k, pf in pfs.items()}).vbt.plot(
    title="Drawdown (%)").show()

In [13]:
cols = ["Total Return [%]", "Max Drawdown [%]", "Sharpe Ratio", "Sortino Ratio",
        "Total Trades", "Win Rate [%]", "Profit Factor"]
comparison = pd.DataFrame({k: pf.stats()[cols] for k, pf in pfs.items()}).T
comparison.astype(float).round(2)

,Total Return [%],Max Drawdown [%],Sharpe Ratio,Sortino Ratio,Total Trades,Win Rate [%],Profit Factor
Buy & hold,809.37,33.72,0.86,1.21,1.0,NaN,NaN
MA 50/200 L/S,71.38,48.76,0.28,0.39,14.0,38.46,1.28
RSI long,225.19,28.90,0.67,0.99,51.0,80.39,3.69
Bollinger L/S,44.20,31.94,0.23,0.34,154.0,66.01,1.27


## 7 · Optimizarea parametrilor și overfitting-ul

vectorbt poate testa sute de combinații de parametri dintr-o dată. Testăm toate perechile de medii mobile (rapidă, lentă) dintre 10 și 200 de zile, **doar pe perioada in-sample** (2010–2019).

In [14]:
windows = np.arange(10, 201, 10)
fast_g, slow_g = vbt.MA.run_combs(close, window=windows, r=2, short_names=["fast", "slow"])
long_in = fast_g.ma_crossed_above(slow_g)
short_in = fast_g.ma_crossed_below(slow_g)

pf_grid_is = backtest(long_in, short_in, short_in, long_in, mask=in_sample)
sharpe_is = pf_grid_is.sharpe_ratio()
print(f"{len(sharpe_is)} combinații testate")
sharpe_is.vbt.heatmap(x_level="fast_window", y_level="slow_window",
                      trace_kwargs=dict(colorscale="RdYlGn")).update_layout(
    title="Sharpe in-sample (2010–2019) pe perechi de medii mobile").show()

190 combinații testate


Alegem cea mai bună combinație din **trecut** și o rulăm pe **viitor** (out-of-sample). Comparăm cu regula „din manual” 50/200, aleasă fără optimizare.

In [15]:
best = sharpe_is.idxmax()
pf_grid_oos = backtest(long_in, short_in, short_in, long_in, mask=out_sample)
sharpe_oos = pf_grid_oos.sharpe_ratio()

summary = pd.DataFrame({
    "Sharpe in-sample": [sharpe_is[best], sharpe_is[(50, 200)]],
    "Sharpe out-of-sample": [sharpe_oos[best], sharpe_oos[(50, 200)]],
}, index=[f"cea mai bună in-sample {tuple(int(x) for x in best)}", "regula clasică (50, 200)"]).round(2)
summary.loc["buy & hold"] = [
    vbt.Portfolio.from_holding(close[in_sample], fees=FEES, freq="1D").sharpe_ratio(),
    vbt.Portfolio.from_holding(close[out_sample], fees=FEES, freq="1D").sharpe_ratio()]
summary.round(2)

,Sharpe in-sample,Sharpe out-of-sample
"cea mai bună in-sample (30, 180)",0.63,0.47
"regula clasică (50, 200)",0.38,0.42
buy & hold,0.92,0.80


In [16]:
pd.Series(sharpe_oos.values, index=sharpe_is.values, name="combinație").sort_index().vbt.scatterplot(
    ).update_layout(title="Fiecare punct = o combinație de medii mobile",
                    xaxis_title="Sharpe in-sample (2010–2019)",
                    yaxis_title="Sharpe out-of-sample (2020 → azi)").show()
rank_oos = int(sharpe_oos.rank(ascending=False)[best])
print(f"Corelația dintre Sharpe in-sample și out-of-sample: "
      f"{np.corrcoef(sharpe_is.values, sharpe_oos.values)[0, 1]:.2f}")
print(f"Combinația nr. 1 in-sample {tuple(int(x) for x in best)} ajunge pe locul {rank_oos} "
      f"din {len(sharpe_oos)} out-of-sample; Sharpe {sharpe_is[best]:.2f} → {sharpe_oos[best]:.2f}")

Corelația dintre Sharpe in-sample și out-of-sample: 0.12
Combinația nr. 1 in-sample (30, 180) ajunge pe locul 83 din 190 out-of-sample; Sharpe 0.63 → 0.47


**Lecția:** Sharpe-ul combinației câștigătoare scade când trece pe date noi, iar corelația dintre clasamentul din trecut și cel din viitor este slabă. Un clasament bun în trecut prezice puțin despre viitor. Dacă testezi suficiente variante, una va arăta bine *din întâmplare*. Și nicio variantă de medii mobile nu a bătut buy & hold, nici în trecut, nici în viitor. Acesta este **overfitting-ul** (supra-ajustarea): ai învățat zgomotul, nu un tipar real. Regulă practică: alegi parametrii pe in-sample, îi verifici **o singură dată** pe out-of-sample și raportezi câte variante ai încercat.

## 8 · Două greșeli clasice

### a) Execuție în aceeași zi
Calculăm RSI cu prețul de închidere de azi și „cumpărăm” tot la închiderea de azi. În realitate nu poți afla prețul de închidere și tranzacționa la el în același timp, deci rezultatul raportat este unul pe care nu l-ai fi putut obține.

In [17]:
same_bar = vbt.Portfolio.from_signals(close, rsi_entries, rsi_exits, fees=FEES,
                                      slippage=SLIPPAGE, init_cash=INIT_CASH, freq="1D")
pd.DataFrame({
    "corect: semnal azi → execuție mâine la deschidere": pf_rsi.stats()[cols],
    "greșit: semnal și execuție la aceeași închidere": same_bar.stats()[cols],
}).astype(float).round(2)

,corect: semnal azi → execuție mâine la deschidere,greșit: semnal și execuție la aceeași închidere
Total Return [%],225.19,200.79
Max Drawdown [%],28.90,28.34
Sharpe Ratio,0.67,0.63
Sortino Ratio,0.99,0.93
Total Trades,51.00,51.00
Win Rate [%],80.39,82.35
Profit Factor,3.69,3.63


Diferența poate fi în orice direcție. Aici execuția „greșită” iese chiar puțin mai slabă, pentru că după o zi de panică piața deschide adesea mai jos și varianta corectă cumpără mai ieftin. Problema nu este că rezultatul e mai mare sau mai mic, ci că **nu este realizabil**. Pe strategii rapide sau pe date intraday, aceeași greșeală poate umfla rezultatele foarte mult.

### b) Privitul în viitor (look-ahead)
O strategie care „știe” randamentul de mâine: long dacă mâine piața crește, short dacă scade. E un bug evident aici, dar în cod real apare discret, de exemplu printr-un `shift(-1)` greșit sau o medie calculată pe toată seria, inclusiv pe viitor.

In [18]:
tomorrow_up = close.pct_change().shift(-1) > 0          # uses FUTURE data -> cheating
cheat = vbt.Portfolio.from_signals(close, tomorrow_up, ~tomorrow_up,
                                   short_entries=~tomorrow_up, short_exits=tomorrow_up,
                                   fees=FEES, init_cash=INIT_CASH, freq="1D")
print(f"Sharpe 'perfect': {cheat.sharpe_ratio():.1f}   |   Total return: {cheat.total_return()*100:,.0f}%")

Sharpe 'perfect': 13.3   |   Total return: 110,059,993,669,180%


Un Sharpe de zeci și randamente astronomice nu înseamnă că ai descoperit ceva. Înseamnă aproape sigur că ai un bug. **Dacă un backtest arată prea bine ca să fie adevărat, de obicei nu este.**

## 9 · Exerciții

1. Schimbă perioadele mediilor mobile în strategia 1 (de exemplu 20/100). Ce se întâmplă cu numărul de tranzacții și cu costurile?
2. Pune costurile la zero (`FEES = SLIPPAGE = 0`). Care strategie este cea mai afectată de costuri și de ce?
3. Strategia RSI: încearcă praguri 20/50 sau 35/60. Cum se schimbă Win Rate și Max Drawdown?
4. Înlocuiește `SPY` cu `QQQ`, `GLD` sau `BTC-USD`. Funcționează mai bine strategiile de trend sau cele de mean-reversion?
5. Fă strategia 1 doar long (scoate `short_entries` / `short_exits`). E mai bună decât varianta long/short?